# Screening Task: Quantum States, Eigenvalues and the Rydberg Blockade

You do **not** need any prior quantum mechanics. This notebook teaches the small amount of linear algebra and quantum language you need, and you will then use it to simulate a blockade yourself. All the physics you need is stated in the text.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

np.set_printoptions(precision=4, suppress=True)

---
# Part 0: States as vectors (~15 min)

## Reading
Consider an atom reduced to two levels: a **ground state** $|g\rangle$ and an **excited (Rydberg) state** $|r\rangle$. The symbol $|\cdot\rangle$ ("ket") just means "this is a quantum state". We represent them as vectors:

$$|g\rangle = \begin{pmatrix}1\\0\end{pmatrix},\qquad |r\rangle = \begin{pmatrix}0\\1\end{pmatrix}.$$

A general state is a **superposition**
$$|\psi\rangle = c_g|g\rangle + c_r|r\rangle = \begin{pmatrix}c_g\\c_r\end{pmatrix},$$
where the numbers $c_g, c_r$ are **complex** and are called *amplitudes*.

Two rules connect this to measurements:

1. **Probabilities:** if you measure the atom, the probability of finding it in $|r\rangle$ is $P_r = |c_r|^2$. Similarly $P_g = |c_g|^2$.
2. **Normalisation:** the probabilities must add up to 1, so $|c_g|^2 + |c_r|^2 = 1$.

The **inner product** of two states is $\langle a|b\rangle = \sum_i a_i^* b_i$. In numpy this is `np.vdot(a, b)`, which complex-conjugates the first argument. Note that $\langle\psi|\psi\rangle = |c_g|^2+|c_r|^2$ is the squared length of the vector, so normalisation says the vector has length 1. Two states with $\langle a|b\rangle = 0$ are called **orthogonal**.

In Python, complex numbers are written like `0.8j`, and you must use `dtype=complex` arrays whenever amplitudes may be complex.

### Exercise 0.1
For the state $|\psi\rangle = \begin{pmatrix}0.6\\0.8i\end{pmatrix}$:
1. Check it is normalised (use `np.vdot`).
2. Compute $P_g$ and $P_r$.

In [ ]:
g = np.array([1, 0], dtype=complex)
r = np.array([0, 1], dtype=complex)
psi = np.array([0.6, 0.8j])

# YOUR CODE HERE: print the norm <psi|psi>, P_g and P_r

### Exercise 0.2
Build $|+\rangle = (|g\rangle+|r\rangle)/\sqrt2$ and $|-\rangle = (|g\rangle-|r\rangle)/\sqrt2$.
Print $P_r$ for each, and the inner product $\langle +|-\rangle$.

**Question (1–2 sentences):** the two states give the same measurement probabilities, yet $\langle+|-\rangle = 0$. What does that tell you about the information in the *signs/phases* of amplitudes?

In [ ]:
# YOUR CODE HERE

**Your answer:** *(double-click to edit)*

---
# Part 1: Eigenvalues and eigenvectors

## Reading: matrices act on vectors
A matrix $M$ takes a vector $v$ and returns another vector $Mv$. In general the output points in a *different direction*. For example, the matrix $X=\begin{pmatrix}0&1\\1&0\end{pmatrix}$ swaps the two entries. Run the cell below to see it.

In [ ]:
X = np.array([[0, 1],
              [1, 0]])
for v in ([1, 0], [1, 1], [1, -1]):
    v = np.array(v)
    print(f"v = {v},   X v = {X @ v}")

Look at the output: for $v=(1,0)$ the result $(0,1)$ points somewhere else. But for $v=(1,1)$ the result is $(1,1)$, the same vector, and for $v=(1,-1)$ the result is $-(1,-1)$, again along the same line, just flipped.

Such special vectors are called **eigenvectors** and the stretching factor is the **eigenvalue**:

$$M v = \lambda\, v.$$

Here $(1,1)$ has $\lambda=+1$ and $(1,-1)$ has $\lambda=-1$.

**How to find them.** Rewrite as $(M-\lambda I)v = 0$. A non-zero $v$ can only satisfy this if $\det(M-\lambda I)=0$. For $X$:
$$\det\begin{pmatrix}-\lambda&1\\1&-\lambda\end{pmatrix}=\lambda^2-1=0\;\Rightarrow\;\lambda=\pm1.$$
Plugging each $\lambda$ back in gives the eigenvector.

## Why quantum mechanics cares
In quantum mechanics the energy of a system is described by a matrix $H$, the **Hamiltonian**. Its eigenvalues are the allowed **energies** and its eigenvectors are the **stationary states**, i.e. the states that do not change (apart from a phase) as time passes. We will show this precisely in Part 2.

Hamiltonians are **Hermitian** ($H = H^\dagger$, meaning complex-conjugate transpose). For Hermitian matrices:
- the eigenvalues are real numbers,
- the eigenvectors can be chosen orthonormal (orthogonal, each of length 1).

numpy has a function for exactly this case: `E, V = np.linalg.eigh(H)`. It returns the eigenvalues `E` (sorted, smallest first) and a matrix `V` whose **columns** are the eigenvectors: `V[:, k]` belongs to `E[k]`. Remember "columns", because it is the most common mistake.

## The atom's Hamiltonian
A two-level atom driven by a laser has

$$H=\begin{pmatrix}0&\Omega/2\\ \Omega/2&\Delta\end{pmatrix}\quad\text{in the basis }\{|g\rangle,|r\rangle\}.$$

Here $\Omega$ is the **Rabi frequency** (laser strength, the off-diagonal term that couples $g$ and $r$) and $\Delta$ is the **detuning** (how far the laser is from resonance).

### Exercise 1.1: by hand, then check (resonant case)
Take $\Omega=1$, $\Delta=0$, so $H=\begin{pmatrix}0&0.5\\0.5&0\end{pmatrix}$.

1. **By hand** (write it in the cell below, or on paper and describe it), find the eigenvalues using $\det(H-\lambda I)=0$, then find the eigenvectors.
2. Use `np.linalg.eigh` to check.
3. Check numerically that $Hv=Ev$ for each eigenvector (columns!) and that $V^\dagger V = I$ (orthonormality). Use `np.allclose`.

**Your hand calculation:** *(double-click to edit)*

In [ ]:
Omega = 1.0

def H_atom(Delta, Omega=1.0):
    return np.array([[0, Omega/2],
                     [Omega/2, Delta]], dtype=complex)

H = H_atom(0.0)
# YOUR CODE HERE: eigenvalues/eigenvectors with eigh, then check H v = E v and V^dagger V = I

---
# Time evolution using eigenvectors

## Reading
How does a state change in time? Quantum mechanics says
$$i\frac{d}{dt}|\psi(t)\rangle = H|\psi(t)\rangle\qquad(\hbar=1).$$

**Special case: an eigenvector.** If $|\psi(0)\rangle=v_k$ with $Hv_k=E_kv_k$, the solution is
$$|\psi(t)\rangle=e^{-iE_kt}\,v_k.$$
You can check this by substituting it into the equation. The state only picks up a phase, so its measurement probabilities never change. That is why eigenvectors are called *stationary states*.

**General case: superposition.** Any starting state can be written as a combination of the eigenvectors, $|\psi(0)\rangle=\sum_k c_k v_k$, with coefficients $c_k=\langle v_k|\psi(0)\rangle$ (this is where orthonormality helps). Since the equation is linear, each piece evolves with its own phase:
$$|\psi(t)\rangle=\sum_k c_k\,e^{-iE_kt}\,v_k.$$

**Recipe:**
1. diagonalise $H$ to get `E`, `V`,
2. compute $c = V^\dagger\psi_0$ (the coefficients in the eigenbasis),
3. multiply each $c_k$ by $e^{-iE_kt}$,
4. convert back: $\psi(t)=V\,(\text{result})$.

Because the phases $e^{-iE_kt}$ of different pieces rotate at different rates, they *interfere*. That interference produces the oscillation of $P_r(t)$ between 0 and a maximum, called **Rabi oscillations**.

(For comparison, the same thing is written $|\psi(t)\rangle=e^{-iHt}|\psi(0)\rangle$ with a "matrix exponential", which `scipy.linalg.expm` computes directly. We will use it as a cross-check.)